# Agentic AI LS26 — Week 2 Assignment

**Topic**: RAG Pipelines, Chunking Strategies & RAG vs Fine-Tuning

In [ ]:
# All imports
import re
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

## Load Input Document

In [ ]:
# Load the text file
with open("input_text.txt", "r", encoding="utf-8") as f:
    text = f.read()
print(f"Document loaded: {len(text.split())} words")

---
# Question 1: Build a Minimal RAG Pipeline from Scratch

Implement the core RAG pipeline without using any high-level RAG framework.
Everything: chunking, embedding, vector search, and generation is wired manually.

## Part A: Chunking
Implement fixed-size chunking with overlap.

In [ ]:
def chunk_text(text: str, chunk_size: int = 200, overlap: int = 40) -> list[str]:
    """
    Split `text` into chunks of ~chunk_size words,
    with `overlap` words shared between consecutive chunks.
    Returns a list of chunk strings.
    """
    words = text.split()
    result = []
    start = 0
    step = chunk_size - overlap

    while start < len(words):
        end = start + chunk_size
        chunk = " ".join(words[start:end])
        result.append(chunk)
        start += step

    return result

# Chunk the document and print total number
chunks = chunk_text(text)
print(f"Total number of chunks: {len(chunks)}")

## Part B: Embedding & Vector Store

- Embed chunks using `sentence-transformers` (model: `all-MiniLM-L6-v2`)
- Store embeddings in a numpy array
- Implement cosine similarity manually (no sklearn)
- Run 3 queries and print top-3 retrieved chunks with cosine scores

In [ ]:
# Load embedding model and embed all chunks
print("Loading embedding model (all-MiniLM-L6-v2)...")
model = SentenceTransformer("all-MiniLM-L6-v2")
chunk_embeddings = model.encode(chunks)
print(f"Embedded {len(chunks)} chunks -> shape {chunk_embeddings.shape}")

In [ ]:
def cosine_similarity(a, b):
    """
    Compute cosine similarity: cos(theta) = A.B / ||A|| ||B||
    (No sklearn used.)
    """
    dot_product = np.dot(a, b)
    norm_a = np.linalg.norm(a)
    norm_b = np.linalg.norm(b)
    if norm_a == 0 or norm_b == 0:
        return 0.0
    return float(dot_product / (norm_a * norm_b))


def retrieve(query: str, top_k: int = 3) -> list:
    """Return the top_k most similar chunks to `query` with cosine scores."""
    query_embedding = model.encode(query)
    scores = [cosine_similarity(query_embedding, ce) for ce in chunk_embeddings]
    top_indices = np.argsort(scores)[::-1][:top_k]
    return [(chunks[i], float(scores[i])) for i in top_indices]

In [ ]:
# Run 3 different queries and print top-3 retrieved chunks
queries_b = [
    "What are the elements of emotional intelligence?",
    "Can leadership be trained or is it innate?",
    "What is the role of feedback in leadership training?",
]

for q in queries_b:
    print(f"\nQuery: {q}")
    print("-" * 55)
    results = retrieve(q, top_k=3)
    for rank, (chunk, score) in enumerate(results, 1):
        preview = " ".join(chunk.split()[:30]) + " ..."
        print(f"  Top-{rank} [cosine score = {score:.4f}]: {preview}")

## Part C: Generation

- Use `google/flan-t5-base` via HuggingFace
- Wire the retriever into an LLM call
- Run 2 queries: 1 answerable + 1 out-of-scope (hallucination guard)

In [ ]:
# Load generation model
print("Loading generation model (google/flan-t5-base)...")
tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-base")
gen_model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-base")
print("Generator ready.")

In [ ]:
def rag_answer(query: str) -> str:
    """Retrieve relevant chunks and generate an answer using the LLM."""
    results = retrieve(query, top_k=3)
    context = "\n\n".join(chunk for chunk, _ in results)
    prompt = f"""Answer the question using ONLY the context below.
If the answer is not in the context, say 'I don't know'.

Context:
{context}

Question: {query}
Answer:"""
    inputs = tokenizer(prompt, return_tensors="pt", max_length=512, truncation=True)
    outputs = gen_model.generate(**inputs, max_new_tokens=256)
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

In [ ]:
# Answerable query
q_in = "What are the five elements of emotional intelligence?"
print(f"Query (answerable): {q_in}")
print(f"Answer: {rag_answer(q_in)}")

print()

# Out-of-scope query (hallucination guard)
q_out = "What is the chemical formula for photosynthesis?"
print(f"Query (out-of-scope): {q_out}")
print(f"Answer: {rag_answer(q_out)}")

---
# Question 2: Chunking Strategy Showdown

Comparing three chunking strategies and measuring their effect on retrieval quality.

## Part A: Implement Three Chunkers
Each function returns a list of strings.

In [ ]:
def fixed_chunk(text: str, size: int = 300, overlap: int = 50) -> list[str]:
    """Fixed-size word-level chunking with overlap."""
    words = text.split()
    chunks_list = []
    start = 0
    step = size - overlap
    while start < len(words):
        chunk = " ".join(words[start : start + size])
        chunks_list.append(chunk)
        start += step
    return chunks_list


def sentence_chunk(text: str) -> list[str]:
    """
    Split at sentence boundaries ('. ', '? ', '! ').
    Group sentences into chunks of ~5 sentences each with 1-sentence overlap.
    """
    sentences = re.split(r'(?<=[.?!])\s+', text)
    sentences = [s.strip() for s in sentences if s.strip()]
    chunks_list = []
    group_size = 5
    overlap = 1
    start = 0
    while start < len(sentences):
        end = min(start + group_size, len(sentences))
        chunk = " ".join(sentences[start:end])
        chunks_list.append(chunk)
        if end >= len(sentences):
            break
        start += group_size - overlap
    return chunks_list


def sliding_window_chunk(text: str, window: int = 400, step: int = 100) -> list[str]:
    """Each chunk is `window` words, starting every `step` words (heavy overlap)."""
    words = text.split()
    chunks_list = []
    start = 0
    while start < len(words):
        end = min(start + window, len(words))
        chunk = " ".join(words[start:end])
        chunks_list.append(chunk)
        if end >= len(words):
            break
        start += step
    return chunks_list

In [ ]:
# Run all three strategies and print statistics
strategies = {
    "Fixed-size":     fixed_chunk(text),
    "Sentence-based": sentence_chunk(text),
    "Sliding window":  sliding_window_chunk(text),
}

print("=" * 70)
print("Chunking Strategy Statistics")
print("=" * 70)
header = f"{'Strategy':<20} | {'Chunks':>6} | {'Mean Len (wds)':>14} | {'Min Len':>7} | {'Max Len':>7}"
print(header)
print("-" * 70)

for name, chnks in strategies.items():
    lengths = [len(c.split()) for c in chnks]
    print(
        f"{name:<20} | {len(chnks):>6} | {np.mean(lengths):>14.1f} | "
        f"{min(lengths):>7} | {max(lengths):>7}"
    )

## Part B: Retrieval Comparison

- Build a separate retriever for each chunking strategy
- 5 manually written question-answer pairs
- Hit Rate = questions where answer found in top-3 / 5

In [ ]:
def retrieve_top_k(query, chnks, embeddings, top_k=3):
    """Return top-k (chunk, score) tuples for a query."""
    q_emb = model.encode(query)
    scores = [cosine_similarity(q_emb, e) for e in embeddings]
    top_idx = np.argsort(scores)[::-1][:top_k]
    return [(chnks[i], float(scores[i])) for i in top_idx]

# 5 manually written QA pairs from the document
qa_pairs = [
    (
        "What are the five elements of emotional intelligence?",
        "self-awareness, self-regulation, motivation, empathy, and social skills",
    ),
    (
        "Who advanced the concept of emotional intelligence in leadership?",
        "Daniel Goleman",
    ),
    (
        "Which company started a Training Facility at an affiliate college?",
        "Bentley Motors",
    ),
    (
        "How much is spent on training managers every year in the United Kingdom?",
        "50 billion dollars",
    ),
    (
        "What leadership styles did Kurt Lewin identify?",
        "autocratic, democratic, and Laissez-faire",
    ),
]

# Evaluate each strategy
results_table = {}
for name, chnks in strategies.items():
    print(f"Embedding chunks for '{name}' strategy...")
    embeddings = model.encode(chnks)
    hits = 0
    for question, answer in qa_pairs:
        top3 = retrieve_top_k(question, chnks, embeddings, top_k=3)
        found = any(answer.lower() in chunk.lower() for chunk, _ in top3)
        if found:
            hits += 1
    results_table[name] = hits

# Print comparison table
print(f"\n{'Strategy':<20} | {'Chunks':>6} | {'Mean Len':>10} | {'Hit Rate':>10}")
print("-" * 60)
for name, chnks in strategies.items():
    lengths = [len(c.split()) for c in chnks]
    hr = results_table[name]
    print(
        f"{name:<20} | {len(chnks):>6} | "
        f"{np.mean(lengths):>7.0f} wds | "
        f"    {hr}/5"
    )

---
# Question 3: RAG vs Fine-Tuning: Decision and Demo

Part A: Decision tree for choosing between RAG, Fine-Tuning, or both.
Part B: Hallucination stress test with bar chart.

## Part A: Build the Decision Tree

In [ ]:
def recommend_approach(scenario: dict) -> str:
    """
    Input keys:
      - data_changes_frequently: bool
      - need_specific_output_format: bool
      - budget: str  # 'low' | 'medium' | 'high'
      - latency_sensitive: bool
      - knowledge_type: str  # 'factual' | 'behavioral' | 'both'
    Returns one of: 'RAG', 'Fine-Tuning', 'RAG + Fine-Tuning',
    'Prompt Engineering only' with a 1-sentence justification.
    """
    data_changes = scenario["data_changes_frequently"]
    format_need  = scenario["need_specific_output_format"]
    budget       = scenario["budget"]
    latency      = scenario["latency_sensitive"]
    knowledge    = scenario["knowledge_type"]

    # Both factual + behavioural needs => combine
    if knowledge == "both":
        if budget == "low":
            return ("RAG + Fine-Tuning",
                    "Both factual retrieval and behavioural adaptation are needed; "
                    "even on a tight budget the combination is essential for quality.")
        return ("RAG + Fine-Tuning",
                "Combining RAG for dynamic factual knowledge with Fine-Tuning "
                "for specialised output behaviour yields the best results.")

    # Factual knowledge
    if knowledge == "factual":
        if data_changes:
            return ("RAG",
                    "Frequently changing factual data is best served by RAG's "
                    "dynamic retrieval without costly retraining.")
        if budget == "low" and not format_need and not latency:
            return ("Prompt Engineering only",
                    "Static factual needs with a low budget and no special "
                    "format requirements can be addressed through careful prompt design.")
        return ("RAG",
                "Factual knowledge retrieval is RAG's core strength, providing "
                "grounded answers from a document store.")

    # Behavioural knowledge
    if knowledge == "behavioral":
        if format_need and budget != "low":
            return ("Fine-Tuning",
                    "Specific output formats and behavioural patterns require "
                    "fine-tuning the model on curated examples.")
        if budget == "low" and not format_need:
            return ("Prompt Engineering only",
                    "Simple behavioural adjustments on a low budget are best "
                    "handled through well-crafted prompt engineering.")
        return ("Fine-Tuning",
                "Behavioural adaptation - tone, style, and format - is best "
                "achieved through fine-tuning on representative data.")

    # Fallback
    return ("Prompt Engineering only",
            "The requirements are simple enough to be addressed with "
            "prompt engineering alone.")

In [ ]:
# Test on 5 diverse scenarios
scenarios = [
    {
        "name": "Customer-support chatbot with frequently changing FAQs",
        "data_changes_frequently": True,
        "need_specific_output_format": False,
        "budget": "medium",
        "latency_sensitive": True,
        "knowledge_type": "factual",
    },
    {
        "name": "Medical report generator with strict format",
        "data_changes_frequently": False,
        "need_specific_output_format": True,
        "budget": "high",
        "latency_sensitive": False,
        "knowledge_type": "behavioral",
    },
    {
        "name": "Legal research assistant needing citations + specific format",
        "data_changes_frequently": True,
        "need_specific_output_format": True,
        "budget": "high",
        "latency_sensitive": False,
        "knowledge_type": "both",
    },
    {
        "name": "Simple internal Q&A on static company policy (low budget)",
        "data_changes_frequently": False,
        "need_specific_output_format": False,
        "budget": "low",
        "latency_sensitive": False,
        "knowledge_type": "factual",
    },
    {
        "name": "News summarisation service with daily updates",
        "data_changes_frequently": True,
        "need_specific_output_format": False,
        "budget": "medium",
        "latency_sensitive": True,
        "knowledge_type": "factual",
    },
]

for sc in scenarios:
    name = sc["name"]
    recommendation, justification = recommend_approach(sc)
    print(f"\nScenario: {name}")
    print(f"  -> {recommendation}")
    print(f"     {justification}")

## Part B: Hallucination Stress Test

- 6 queries: 3 answerable + 3 out-of-scope
- Log: top chunk (first 100 words), cosine score, LLM answer
- Bar chart: cosine scores color-coded (green = answerable, red = out-of-scope)

In [ ]:
# 3 answerable queries (answers exist in the leadership document)
answerable = [
    "What are the five elements of emotional intelligence?",
    "Which company started a Training Facility at an affiliate college?",
    "What did Plato believe about training a good leader?",
]

# 3 out-of-scope queries (plausible-sounding but factually absent)
out_of_scope = [
    "What is the chemical formula for photosynthesis?",
    "Which programming language is best for machine learning?",
    "What are the main causes of climate change?",
]

all_queries = answerable + out_of_scope
labels = ["Answerable"] * len(answerable) + ["Out-of-scope"] * len(out_of_scope)
colors = ["green"] * len(answerable) + ["red"] * len(out_of_scope)

scores = []
for i, query in enumerate(all_queries):
    results = retrieve(query, top_k=3)
    top_chunk, top_score = results[0]
    answer = rag_answer(query)

    # First 100 words of top chunk
    top_chunk_preview = " ".join(top_chunk.split()[:100])

    print(f"\n{'~' * 70}")
    print(f"Query {i+1} [{labels[i]}]: {query}")
    print(f"Top chunk (first 100 words): {top_chunk_preview}")
    print(f"Cosine similarity score:     {top_score:.4f}")
    print(f"LLM answer:                  {answer}")

    scores.append(top_score)

In [ ]:
# Bar chart: cosine similarity scores color-coded by query type
fig, ax = plt.subplots(figsize=(10, 6))
x = np.arange(len(all_queries))
bars = ax.bar(x, scores, color=colors, edgecolor="black", linewidth=0.8)

# Add score labels on bars
for bar, score in zip(bars, scores):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01,
            f"{score:.3f}", ha="center", va="bottom", fontsize=9)

short_labels = [f"Q{i+1}: " + (q[:35] + "..." if len(q) > 35 else q)
                for i, q in enumerate(all_queries)]
ax.set_xticks(x)
ax.set_xticklabels(short_labels, rotation=30, ha="right", fontsize=8)
ax.set_ylabel("Cosine Similarity Score")
ax.set_title("Hallucination Stress Test: Cosine Scores by Query Type")
ax.set_ylim(0, max(scores) + 0.15)

legend_elements = [
    Patch(facecolor="green", edgecolor="black", label="Answerable"),
    Patch(facecolor="red",   edgecolor="black", label="Out-of-scope"),
]
ax.legend(handles=legend_elements, loc="upper right")

plt.tight_layout()
plt.savefig("hallucination_stress_test.png", dpi=150)
plt.show()

In [ ]:
# Analysis
avg_answerable = np.mean(scores[:len(answerable)])
avg_oos = np.mean(scores[len(answerable):])

print("=" * 70)
print("ANALYSIS")
print("=" * 70)
print(f"Average cosine score (answerable):   {avg_answerable:.4f}")
print(f"Average cosine score (out-of-scope): {avg_oos:.4f}")

if avg_answerable > avg_oos:
    print(
        "\nAs expected, answerable queries have HIGHER similarity scores,\n"
        "confirming the retriever surfaces relevant chunks for in-scope queries.\n"
        "Lower scores for out-of-scope queries help the grounding prompt\n"
        "produce 'I don\'t know' answers, acting as a hallucination guard."
    )
else:
    print(
        "\nUnexpected: out-of-scope scores are similar to or higher than\n"
        "answerable scores. This may indicate the grounding prompt needs\n"
        "strengthening or the document has incidental overlap with\n"
        "out-of-scope topics."
    )